# Beyond Ironclad: Fact-Checking the Data Story

This notebook offers a short guide and tutorial on how to identify and fix common errors in data projects. While the notebook is written in Python, the lessons apply across the board.



In [ ]:
!git clone https://github.com/ethanscorey/nicar-beyond-ironclad-2026-source-materials.git
%cd nicar-beyond-ironclad-2026-source-materials

Cloning into 'nicar-beyond-ironclad-2026-source-materials'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 47 (delta 19), reused 11 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 11.87 KiB | 11.87 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/nicar-beyond-ironclad-2026-source-materials/nicar-beyond-ironclad-2026-source-materials/nicar-beyond-ironclad-2026-source-materials


In [ ]:
!pip install pandas numpy

In [ ]:

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## Loading the Data

In [ ]:
transit = pd.read_csv("data/mass_transit_delays.csv")
transit_dict = pd.read_csv(
    "data/mass_transit_delays_dictionary.csv",
    index_col="column_name",
)

counties = pd.read_csv("data/county_population.csv", dtype={"county_fips": str})
air = pd.read_csv("data/county_air_pollution.csv", dtype={"county_fips": str})
counties_dict = pd.read_csv(
    "data/county_population_dictionary.csv",
    index_col="column_name",
)
air_dict = pd.read_csv(
    "data/county_air_pollution_dictionary.csv",
    index_col="column_name",
)

income = pd.read_csv("data/income_demographics.csv")
income_dict = pd.read_csv(
    "data/income_demographics_dictionary.csv",
    index_col="column_name",
)

bookings = pd.read_csv("data/bookings.csv")
bookings_dict = pd.read_csv(
    "data/bookings_data_dictionary.csv",
    index_col="column_name",
)

race = pd.read_csv("data/race_ethnicity_versions.csv")
race_dict = pd.read_csv(
    "data/race_ethnicity_versions_dictionary.csv",
    index_col="column_name",
)

print("Datasets loaded.")

Datasets loaded.


In [ ]:
print('Mass transit dictionary')
display(transit_dict)

print('County population dictionary')
display(counties_dict)

print('County air pollution dictionary')
display(air_dict)

print('Income demographics dictionary')
display(income_dict)

print('Race/ethnicity dictionary')
display(race_dict)

Mass transit dictionary


,data_type,description,notes
column_name,,,
route_id,text,Unique mass transit route identifier,Primary key with month
month,date (YYYY-MM),Reporting month,One row per route per month
scheduled_trips,text numeric,Number of trips planned in that month,Contains commas as thousands separators
completed_trips,text numeric,Number of trips actually completed in that month,Contains commas as thousands separators
avg_delay_minutes,float,Average delay in minutes for completed trips,Already numeric
ridership,text numeric,Estimated total riders in that month,Contains commas as thousands separators


County population dictionary


,data_type,description,notes
column_name,,,
county_fips,text,5-digit county FIPS code,Leading zeros must be preserved
county_name,text,County name,Human-readable label
state,text,2-letter USPS state abbreviation,Used for filtering or grouping
population,integer,Estimated county population,Used as denominator or weight


County air pollution dictionary


,data_type,description,notes
column_name,,,
county_fips,text,5-digit county FIPS code,Join key to county population table
year,integer,Calendar year for air pollution measurements,One year included in this demo
pm25_annual,float,Annual average PM2.5 concentration (µg/m³),Higher values indicate poorer air quality
ozone_days_unhealthy,integer,Days in year with unhealthy ozone levels,Simplified count metric


Income demographics dictionary


,data_type,description,notes
column_name,,,
state,text,State abbreviation for each row,Multiple rows exist per income bracket because...
income_bracket,text,Household income category,Repeat categories across states to support gro...
households,integer,Number of households in bracket within state,Useful context column
white_residents,integer,Residents identifying as White alone (non-Hisp...,Numerator for percentage calculation
total_residents,integer,Total residents in bracket and state,Denominator for weighted calculations


Race/ethnicity dictionary


,data_type,description,notes
column_name,,,
region,text,Geographic reporting area,One row per district
total_population,integer,Total district population denominator,Use this to calculate percentages
b02001_002e,integer,ACS B02001_002E White alone count,Race category includes people of Hispanic ethn...
b02001_003e,integer,ACS B02001_003E Black or African American alon...,Race category includes people of Hispanic ethn...
b02001_004e,integer,ACS B02001_004E American Indian and Alaska Nat...,Race category includes people of Hispanic ethn...
b02001_005e,integer,ACS B02001_005E Asian alone count,Race category includes people of Hispanic ethn...
b02001_006e,integer,ACS B02001_006E Native Hawaiian and Other Paci...,Race category includes people of Hispanic ethn...
b02001_007e,integer,ACS B02001_007E Some other race alone count,Race category includes people of Hispanic ethn...
b02001_008e,integer,ACS B02001_008E Two or more races count,Race category includes people of Hispanic ethn...


## Exercise 1: Investigating a Suspicious Calculation

A reporter wants to find the total number of completed trips on each transit line. However, the completed trips column is stored as a string, not numeric data, so they convert the data to numeric values first:

In [ ]:
(
    transit
    .assign(
        completed_trips_num=(
            pd.to_numeric(transit["completed_trips"], errors="coerce")
        ),
    )
    .groupby("route_id")
    .agg({"completed_trips_num": "sum"})
)

,completed_trips_num
route_id,
R1,0.0
R2,0.0
R3,0.0
R4,0.0


What went wrong? Try to diagnose it yourself below.

When you're ready to check your work, keep scrolling to find out.

In [ ]:
transit

,route_id,month,scheduled_trips,completed_trips,avg_delay_minutes,ridership
0,R1,2025-01,"12,500","12,230",4.8,"1,245,000"
1,R1,2025-02,"11,900","11,610",5.2,"1,180,500"
2,R2,2025-01,"10,200","9,950",6.9,"980,300"
3,R2,2025-02,"9,850","9,620",7.1,"945,100"
4,R3,2025-01,"8,600","8,420",3.4,"760,000"
5,R3,2025-02,"8,200","8,050",3.2,"740,450"
6,R4,2025-01,"7,400","7,100",8.6,"625,900"
7,R4,2025-02,"7,050","6,720",9.1,"600,120"


Keep scrolling...

Keep scrolling...

Keep scrolling...

Keep scrolling...

Keep scrolling...

Keep scrolling...

Keep scrolling...

The root cause lies in this line:
```
pd.to_numeric(transit["completed_trips"], errors="coerce")
```

If you check the documentation for [`pd.to_numeric`](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html), you'll see that when the `errors` parameter is set to `"coerce"`, any strings that can't be parsed as valid numbers get converted to `NaN`. The raw data uses commas in the thousands separater, and pandas doesn't know how to parse numbers with thousands separators.


To fix it, we have to clean up the strings first:

In [ ]:
(
    transit
    .assign(
        completed_trips_num=(
            pd.to_numeric(transit["completed_trips"].str.replace(",", ""))
        ),
    )
    .groupby("route_id")
    .agg({"completed_trips_num": "sum"})
)

,completed_trips_num
route_id,
R1,23840
R2,19570
R3,16470
R4,13820


Note that we removed `errors="coerce"` altogether. In general, we want errors to be noisy, because they alert us to problems with our data we might not otherwise notice. For example, if the reporter had simply run the original code without `errors="coerce"`, they likely would have noticed the problem up front:

In [ ]:
(
    transit
    .assign(
        completed_trips_num=(
            pd.to_numeric(transit["completed_trips"])
        ),
    )
    .groupby("route_id")
    .agg({"completed_trips_num": "sum"})
)

ValueError: Unable to parse string "12,230" at position 0

## Exercise 2: Joining Issues

It's very common to want to combine together data from two different datasets; it's a great way to enrich your data and draw connections between sources.

However, it comes with pitfalls as well. In this example, our reporter joined together air pollution data with county population data, using county FIPS codes as the join key:

In [ ]:
joined_buggy = counties.merge(air, on='county_fips', how='inner')
joined_buggy

,county_fips,county_name,state,population,year,pm25_annual,ozone_days_unhealthy
0,01001,Autauga County,AL,59200,2024,9.1,11
1,01003,Baldwin County,AL,247100,2024,8.4,9
2,01005,Barbour County,AL,25200,2024,10.3,13
3,01009,Blount County,AL,59800,2024,8.9,8
4,01013,Butler County,AL,19600,2024,9.8,12


Looks great, right? However, if we compare it with the original dataset, you may notice a problem:

In [ ]:
print("Original row count:", len(counties))
print("Joined row count:", len(joined_buggy))

Original row count: 8
Joined row count: 5


Three of our counties have disappeared!

What went wrong? Try to diagnose it yourself below.

When you're ready to check your work, keep scrolling to find out.

Keep scrolling...

Keep scrolling...

Keep scrolling...

Keep scrolling...

Keep scrolling...

Keep scrolling...

Keep scrolling...

If we compare the county FIPS codes in the `counties` dataset with the county FIPS codes in the `air` dataset, you'll see that three counties have no corresponding air pollution data.

In [ ]:
missing_fips = set(counties['county_fips']) - set(air['county_fips'])
counties[counties['county_fips'].isin(missing_fips)]

,county_fips,county_name,state,population
3,01007,Bibb County,AL,22300
5,01011,Bullock County,AL,10100
7,01015,Calhoun County,AL,116400


To address this, we should use a [left join](https://stackoverflow.com/questions/13997365/sql-joins-as-venn-diagram) instead, which ensures that each row in the `counties` dataset is preserved, even if it has no counterpart in the `air` data. That allows us to choose how to handle the missing data later in the process, rather than it silently slipping away during the join operation.

In [ ]:
joined_left = counties.merge(air, on='county_fips', how='left')
joined_left

,county_fips,county_name,state,population,year,pm25_annual,ozone_days_unhealthy
0,01001,Autauga County,AL,59200,2024.0,9.1,11.0
1,01003,Baldwin County,AL,247100,2024.0,8.4,9.0
2,01005,Barbour County,AL,25200,2024.0,10.3,13.0
3,01007,Bibb County,AL,22300,NaN,NaN,NaN
4,01009,Blount County,AL,59800,2024.0,8.9,8.0
5,01011,Bullock County,AL,10100,NaN,NaN,NaN
6,01013,Butler County,AL,19600,2024.0,9.8,12.0
7,01015,Calhoun County,AL,116400,NaN,NaN,NaN


## Exercise 3: Aggregation Headaches

As we saw in exercise 1, grouping and aggregating data can be a great way to summarize a dataset in a digestible way. However, not all data lends itself to easy aggregation.

In this example, our reporter is curious about racial inequalities in household income, so they decide to compare the percentage of people in each income bracket who self-identify as white. However, the source data is broken up by state, so the reporter decides to group the data by income bracket, then calculate the average percentage of the population that identifies as white for each bracket.

In [ ]:
income

,state,income_bracket,households,white_residents,total_residents
0,CA,Less than $40k,14000,7000,32000
1,TX,Less than $40k,10000,8600,10000
2,CA,$40k to $80k,9000,9000,18000
3,TX,$40k to $80k,9000,11100,18000
4,CA,$80k to $150k,5000,9000,12000
5,TX,$80k to $150k,4000,6750,9000
6,CA,$150k+,1800,5400,7200
7,TX,$150k+,1200,1800,1800


In [ ]:
(
    income
    .assign(pct_white=100*income["white_residents"] / income["total_residents"])
    .groupby("income_bracket")
    .agg({"pct_white": "mean"})
    .sort_values("pct_white")
)

,pct_white
income_bracket,
Less than $40k,53.937500
$40k to $80k,55.833333
$80k to $150k,75.000000
$150k+,87.500000


In this case, there's no obvious issue with the output, but that's why it's important to check the methodology closely as well.

What went wrong? Try to diagnose it yourself below.

When you're ready to check your work, keep scrolling to find out.

Keep scrolling...

Keep scrolling...

Keep scrolling...

Keep scrolling...

Keep scrolling...

Keep scrolling...

Keep scrolling...

Keep scrolling...

The problem here is that percentages are calculated using each state's total population as the denominator. By using a simple average to aggregate these percentages across states, we are giving tiny North Dakota just as much weight as populous states like California and Texas, both of which have more diverse populations than North Dakota.

Instead, we need to either calculate a population-weighted mean, or else postpone the percentage calculation until _after_ the aggregation step. In my opinion, the second approach is a bit more readable, so that's what we'll use here:

In [ ]:
(
    income
    .groupby("income_bracket")
    .agg({"white_residents": "sum", "total_residents": "sum"})
    .assign(pct_white=lambda df: 100*df["white_residents"] / df["total_residents"])
    .sort_values("pct_white")
)

,white_residents,total_residents,pct_white
income_bracket,,,
Less than $40k,15600,42000,37.142857
$40k to $80k,20100,36000,55.833333
$80k to $150k,15750,21000,75.000000
$150k+,7200,9000,80.000000


One thing worth noting: The calculation above would return NaN if there were any income brackets with zero population, since we can't divide by zero. A defensive coding approach would check for that possibility before doing the division, but I want to explicitly recommend against that kind of defensive coding here.

In this case, it would be very unusual for a state to have zero residents in these income brackets. If we found a state where that appeared to be the case in our data, it should make us question the accuracy of the data itself. As noted above, it is generally better to have your code fail loudly instead of quietly; "gracefully" handling errors increases the odds that false assumptions you've made about the data will go unchecked.

## Exercise 4: Garbage In, Garbage Out

Sometimes, the problem with your analysis doesn't lie in any mistake that you've made; it stems from errors in the data itself.

After all, data doesn't spring forth fully formed from Zeus's forehead; all datasets are the product of human measurements, which inevitably contain some errors.

In this example, our reporter wants to calculate the average length of stay for people incarcerated in the local jail. The reporter obtained booking records for each person released from the jail during calendar year 2024. They then calculated the averge length of stay by finding the mean difference between booking date and release date:

In [ ]:
(
    bookings
    .assign(
        release_date=pd.to_datetime(bookings["release_date"]),
        booking_date=pd.to_datetime(bookings["booking_date"]),
        length_of_stay=lambda df: df["release_date"] - df["booking_date"]
    )
    .agg({"length_of_stay": "mean"})
)

,0
length_of_stay,2134 days 20:00:00


If we take these results at face value, the average length of stay is more than 2,134 days, or nearly six years. Nationally, the average length of stay in jails is typically no more than a few weeks, so something is up. Either our local county is a truly extreme outlier, or something went wrong with our calculation.

What went wrong? Try to diagnose it yourself below.

When you're ready to check your work, keep scrolling to find out.

In [ ]:
bookings

,booking_id,person_id,booking_date,date_of_birth,release_date,charge
0,B1001,P001,2024-01-12,1995-08-03,2024-01-14,Shoplifting
1,B1002,P002,2024-02-22,1988-11-19,2024-02-24,DUI
2,B1003,P003,2024-03-03,2001-06-21,2024-03-03,Probation violation
3,B1004,P004,1989-07-14,1989-07-14,2024-08-01,Reckless driving
4,B1005,P005,2024-04-11,1979-12-05,2024-04-13,Disorderly conduct
5,B1006,P006,2024-05-09,1999-02-17,2024-05-10,Trespassing


Keep scrolling...

Keep scrolling...

Keep scrolling...

Keep scrolling...

Keep scrolling...

Keep scrolling...

Our calculations were airtight, so let's take a closer look at the data itself, checking out the latest release date and the earliest booking date:

In [ ]:
(
    bookings
    .assign(
        release_date=pd.to_datetime(bookings["release_date"]),
        booking_date=pd.to_datetime(bookings["booking_date"]),
    )
    .agg({"release_date": "max", "booking_date": "min"})
)

,0
release_date,2024-08-01
booking_date,1989-07-14


1989?! It would be quite unusual for someone to spend 35 years in jail (rather than in prison). What is going on with this person?

In [ ]:
(
    bookings
    .assign(
        booking_date=pd.to_datetime(bookings["booking_date"]),
    )
    .loc[lambda df: df["booking_date"].dt.year == 1989]
)

,booking_id,person_id,booking_date,date_of_birth,release_date,charge
3,B1004,P004,1989-07-14,1989-07-14,2024-08-01,Reckless driving


We have a true anomaly on our hands! If this data is to be believed, person `P004` spent 35 years in jail for a reckless driving charge they allegedly committed on the day of their birth. While letting a day-old baby get behind the wheel of the car would be reckless, it seems _preeeeeeeeeeetty_ unlikely that this is in fact what happened.

Indeed, when we call up the jail, we learn that correctional officers enter the booking date by hand when admitting someone into the jail. In the case of `P004`, whichever guard was on data entry duty that day hadn't had enough coffee and entered `P004`'s date of birth in the booking date field by accident. Unfortunately, this is the only record of `P004`'s time in jail (or at least it's the only record the jail is willing to give you), so we don't know the actual booking date.

That makes the question of how to fix the analysis a bit more thorny. It's obvious that this particular row is a data entry error, but what if correctional officers made mistakes with entering data in other rows?

There's not much we can do about that (except add a graf into our article calling out this sloppy record-keeping!), but we can correct the error we know about and caveat our findings with the context we've learned through our fact-check.

In [ ]:
(
    bookings
    .loc[bookings["booking_date"] != bookings["date_of_birth"]]  # filter out rows where dob and booking date are the same
    .assign(
        release_date=pd.to_datetime(bookings["release_date"]),
        booking_date=pd.to_datetime(bookings["booking_date"]),
        length_of_stay=lambda df: df["release_date"] - df["booking_date"]
    )
    .agg({"length_of_stay": "mean"})
)

,0
length_of_stay,1 days 09:36:00


## Optional Exercise: The Most Common Error I've Seen

The U.S. Census Bureau has lots of value datasets, most notably the American Community Survey, which captures hundreds of demographic, socioeconomic, health, and other variables from across the U.S.

Understandably, reporters love to use it in their data analyses (as they should!). However, there's one error I've seen in more than a dozen stories I've fact-checked over the past few years.

In this exercise, our reporter has attempted to calculate the percentage of the population in each racial group for each district in our city. They pulled their data from the Census API, so the variable names use the original, somewhat opaque Census variable codes:

In [ ]:
race_pcts = (
    race
    .assign(
        pct_white=100*race["b02001_002e"] / race["total_population"],
        pct_black=100*race["b02001_003e"] / race["total_population"],
        pct_aiana=100*race["b02001_004e"] / race["total_population"],
        pct_asian=100*race["b02001_005e"] / race["total_population"],
        pct_nhpi=100*race["b02001_006e"] / race["total_population"],
        pct_other=100*race["b02001_007e"] / race["total_population"],
        pct_multi=100*race["b02001_008e"] / race["total_population"],
        pct_hispanic=100*race["b03003_003e"] / race["total_population"],
    )
)
race_pcts

,region,total_population,b02001_002e,b02001_003e,b02001_004e,b02001_005e,b02001_006e,b02001_007e,b02001_008e,b03003_003e,pct_white,pct_black,pct_aiana,pct_asian,pct_nhpi,pct_other,pct_multi,pct_hispanic
0,North District,120000,50000,22000,3000,18000,2000,15000,10000,28000,41.666667,18.333333,2.500000,15.000000,1.666667,12.500000,8.333333,23.333333
1,Central District,95000,42000,14000,2500,16000,1500,10000,9000,21000,44.210526,14.736842,2.631579,16.842105,1.578947,10.526316,9.473684,22.105263
2,South District,88000,30000,20000,4000,12000,2500,11000,8500,30000,34.090909,22.727273,4.545455,13.636364,2.840909,12.500000,9.659091,34.090909
3,River District,76000,42000,8000,1500,7000,1000,9000,7500,9000,55.263158,10.526316,1.973684,9.210526,1.315789,11.842105,9.868421,11.842105


At first glance, this might look fine. But notice what happens if you add up the percentages for each district:

In [ ]:
(
    race_pcts[race_pcts.columns[race_pcts.columns.str.contains("^pct_")]]
    .sum(axis=1)
)

,0
0,123.333333
1,122.105263
2,134.090909
3,111.842105


That's not right!

What happened? Try to diagnose the issue below. (Consulting the data dictionary might be a good idea...)

When you're ready to check your answer, scroll down to find out.

In [ ]:
Keep scrolling...

In [ ]:
Keep scrolling...

In [ ]:
Keep scrolling...

In [ ]:
Keep scrolling...

In [ ]:
Keep scrolling...

In [ ]:
Keep scrolling...

In [ ]:
Keep scrolling...

The root cause here is that we're double-counting people who identify as Hispanic. Under the [Census's definition of race & ethnicity](https://www.census.gov/topics/population/race/about.html), Hispanic ethnicity is not mutually exclusive with having another racial identity. That decision reflects the [diversity of the Hispanic population in the U.S.](https://www.pewresearch.org/science/2022/06/14/a-brief-statistical-portrait-of-u-s-hispanics/), but it creates problems when we're trying to calculate percentages and we want to include Hispanic as a separate category.

The good news is that the Census includes columns containing population totals for each racial group _excluding_ people who identify as Hispanic. I'll leave it as an exercise to the reader to find out what those columns are and update the data accordingly.

### A More General Note About Race, Ethnicity, and Other Demographic Variables

While the Census uses self-reported racial and ethnic data, not every dataset operates the same way. For instance, many datasets in the criminal justice system report race/ethnicity as perceived by the officer or administrator entering the data. Sometimes they might ask the person how they identify, but other times they're just making a guess based on someone's appearance regardless of how the person themselves might identify. That's obviously problematic for a number of reasons, but it also creates issues when trying to compare criminal justice data with demographic data derived from the Census.

On top of that, many agencies use racial categories that don't map neatly onto the Census's definitions of race and ethnicity. To pick just one example, California [publishes a dataset on law enforcement stops](https://oag.ca.gov/ab953/board/reports) as part of the Racial Identity and Profiling Act of 2005.

In the RIPA dataset, "South Asian / Middle Eastern" is treated as a separate racial category, despite the fact that using Census definitions, people of Middle Eastern heritage are counted as white and people of South Asian backgrounds are included in the broader Asian category. Because of this, it is impossible to accurately compare the demographics of police stops with the Census demographics for a jurisdiction, especially if it's an area with a relatively high South Asian and/or Middle Eastern population.

Gender/sex data faces similar issues, which has recently become much more salient as the Trump administration [revises government surveys to eliminate "gender ideology."](https://www.govexec.com/management/2026/03/lgbtq-data-disappearing-under-trump-reports-find/411835/?oref=ge-featured-river-top) As a result of updates over the past 12 months, many government datasets no longer report gender at all; they report "biological sex," which might be self-reported data or it might be a government official's guess as to someone's sex, depending on the specific dataset in question.

## Final Takeaways / Fact-Checking Checklist

### Takeaways
- Always read the docs, both for the datasets used in the analysis _and_ for the tools used to analyze the data. You should be able to explain in plain English (or some other human language!) what each step in your analysis is doing and _why_.
- Be explicit about the assumptions you're making about your data and verify them! I like adding explicit `assert` statements in my code.
- Document every decision you make, and use version control tools like git to create an auditable record of your changes.
- Use consistent definitions and hard-code them into your analysis. I like to have a separate `settings.py` (or `settings.R`, `settings.js`, etc., depending on your programming language of choice) file in which I define all of the constants used in my analysis. See, for example, [this analysis of ShotSpotter data](https://github.com/ethanscorey/type-southside-weekly-shotspotter/blob/main/shotspotter/settings.py).
- Get another pair of eyes! If that's not possible, take another look at your code after you've had a good night's sleep. If you spend too much time staring at your own code, you'll quickly develop blind spots and miss obvious mistakes.
- Surprising results should be treated with suspicion. The more unusual your result, the more scrutiny you should give it.
- Stress-test your results. Try tweaking some of the assumptions and definitions used in your initial analysis. How does it change the results?

### Fact-Checking Checklist
- Do final row counts match your expectations?
- Do columns have missing data in unexpected places?
- Do percentages add up to 100%?
- Do columns have expected data types?
- Do I know what each column represents?
- Am I using the correct aggregation methods for my data?
- Did I accidentally parse ZIP codes as numbers? (Also applies to any other numeric codes with leading zeros.)
- Did I use the correct [Coordinate Reference System](https://geos270.github.io/Module2/) when conducting geospatial analyses?
- Do I know how my source datasets were created? Have I considered possible sources of error in the data itself?
- Have I tested my analysis on dummy data and verified that the results match my expectations?


## Feedback?

This was my first time running a session like this, so I imagine some things worked better than others. (Or maybe nothing worked!) If you have questions or feedback about this lesson, please don't hesitate to email me at [ethanscorey@gmail.com](mailto:ethanscorey@gmail.com).